# semantic_search/01 — Aggregate note embeddings into per-patient features

Pools each patient's note embeddings into one feature vector per **feature space**, and writes them
as parquet for the clustering stage.

**Runs after** `1_data/01_preprocessing` and `1_data/03_prediction_datasets` (needs the knitted
embedding metadata + array, and `cohort_df` for the treatment anchor). **Runs before**
`semantic_search/02_cluster`. Nothing downstream of the manuscript pipeline depends on it — this
arm is exploratory and additive.

## The grid: 5 spaces x 2 windows

| Space | Width | Notes pooled |
|---|---|---|
| `clinician` | *d* | progress notes only |
| `imaging` | *d* | imaging reports only |
| `pathology` | *d* | pathology reports only |
| `concat` | 3*d* | the three per-type means side by side |
| `merged` | *d* | every note, `NOTE_TYPE` ignored |

*d* is the model's hidden size (768 for Clinical_ModernBERT), read from the array at runtime rather
than hardcoded.

| Window | Notes included |
|---|---|
| `alltime` | every note the patient has, no anchor |
| `pretreatment` | notes strictly before `first_treatment_date` |

`pretreatment` reproduces the production text cohort
(`continuous_window=False`, `max_note_window=0`) and is the leak-free one to use against `tt_death`.
`alltime` is a cross-sectional descriptor of the whole documented history and **is not** safe
against survival — a patient's later notes are written in the knowledge of how they were doing.

## Two things worth knowing

**Pooling is a plain unweighted mean**, not the production `time_decay_mean` (`decay_param=0.01`).
This arm asks what a patient's notes say *on average*, not what they said most recently.

**`merged` is a second pooling pass, not the average of the three per-type means.** The two differ
whenever note counts are unequal across types, which is almost always. `merged` is note-weighted
(200 imaging + 5 pathology notes gives an imaging-dominated vector); averaging the type-means would
be type-weighted. Note-weighted is the intended "a note is a note" reading.

Each space is **independently complete-cased**: a patient with no pathology note is dropped from
`pathology` and from `concat`, but still appears in `merged` and in the spaces for the types they
do have. So `concat` is the three-way intersection and is materially smaller than the others —
the summary at the bottom reports exactly how much smaller.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "semantic_search").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import config  # noqa: E402
from semantic_search import common  # noqa: E402


def check_inputs(preconditions: list[tuple[str, str]]) -> list[str]:
    """Report presence of each (label, path). Returns the missing labels; never raises."""
    missing = []
    for label, path in preconditions:
        ok = os.path.exists(path)
        if not ok:
            missing.append(label)
        print(f"[{'ok ' if ok else 'MISSING'}] {label:<24} {path}")
    print(f"\n{'All inputs present.' if not missing else str(len(missing)) + ' missing: ' + ', '.join(missing)}")
    return missing


def run_module(module: str, args: list[str] | None = None) -> int:
    """Run a semantic_search stage as a subprocess, streaming its output."""
    cmd = [sys.executable, "-m", module] + (args or [])
    print("$ " + " ".join(cmd) + "\n", flush=True)
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=REPO_ROOT)
    print(f"\nexit={proc.returncode}  elapsed={time.time() - t0:,.1f}s", flush=True)
    return proc.returncode


print(f"repo root:  {REPO_ROOT}")
print(f"data root:  {config.DATA_PATH}")
print(f"this arm:   {config.SEMANTIC_SEARCH_PATH}")

## Configuration

`WINDOWS` and `SPACES` mirror `semantic_search.common`; the pooling strategy and the excluded
pre-2015 year-adjustment columns are module constants in `aggregate_embeddings.py` — change them
there, not here.

This stage is the slow one: `pool_embedding_series_vectorized` iterates Python-side over
`group_by(['DFCI_MRN', 'NOTE_TYPE'])`, once per window plus once more for `merged`. It is
skip-if-exists, so a re-run costs nothing unless `OVERWRITE` is set.

In [ ]:
MODULE = "semantic_search.aggregate_embeddings"

WINDOWS = common.WINDOWS          # ["alltime", "pretreatment"]
OVERWRITE = False                 # True -> rebuild feature files that already exist
LIMIT_MRNS = None                 # int -> debug run on the first N patients only

print(f"windows:   {WINDOWS}")
print(f"spaces:    {common.SPACES}")
print(f"overwrite: {OVERWRITE}")
if LIMIT_MRNS:
    print(f"LIMIT_MRNS={LIMIT_MRNS} -- DEBUG RUN, outputs are not the real cohort")

## Preconditions

The embedding array is the large input. This cell does not raise.

In [ ]:
check_inputs([
    ("note embeddings meta",  os.path.join(config.NOTES_PATH,
                                           "full_clinical_notes_embeddings_metadata.parquet")),
    ("note embeddings array", os.path.join(config.NOTES_PATH,
                                           "full_clinical_notes_embeddings_as_array.npy.zst")),
    ("cohort",                os.path.join(config.SURV_PATH, "cohort_df.parquet")),
])

try:
    import zstandard  # noqa: F401
    print("[ok ] zstandard importable (needed to decompress the embedding array)")
except ImportError:
    print("[MISSING] zstandard not importable - run this on the cluster kernel.")

## Pre-flight: what already exists

Read-only census, so a resumed run shows what it will skip.

In [ ]:
existing = common.available_pairs()
print(f"{len(existing)} / {len(common.SPACES) * len(common.WINDOWS)} feature files present")
for space in common.SPACES:
    marks = " ".join(
        f"{w}={'yes' if (space, w) in existing else 'no '}" for w in common.WINDOWS)
    print(f"  {space:10s} {marks}")

## Run

In [ ]:
args = ["--windows", *WINDOWS]
if OVERWRITE:
    args.append("--overwrite")
if LIMIT_MRNS:
    args += ["--limit-mrns", str(LIMIT_MRNS)]

rc = run_module(MODULE, args)
if rc != 0:
    print("\nStage failed - see the traceback above.")

## Summary

`n_patients` per space is the headline: `concat` is the three-way complete-case intersection and
will be the smallest, `merged` the largest. `median_notes_per_patient` is the documentation volume
feeding each mean — stage 3 tests whether it, rather than semantics, is what the clusters separate on.

In [ ]:
import polars as pl

summary_path = common.result_path("feature_summary")
if os.path.exists(summary_path):
    summary = pl.read_csv(summary_path)
    with pl.Config(tbl_rows=20, tbl_width_chars=160):
        print(summary)

    print("\nCohort sizes by space (alltime):")
    alltime = summary.filter(pl.col("window") == "alltime")
    if alltime.height:
        widest = alltime.get_column("n_patients").max()
        for row in alltime.sort("n_patients", descending=True).iter_rows(named=True):
            bar = "#" * int(40 * row["n_patients"] / widest)
            print(f"  {row['space']:10s} {row['n_patients']:7,d}  {bar}")
else:
    print(f"No summary at {summary_path} - has the run completed?")

## Next

`semantic_search/02_cluster.ipynb` clusters each of these feature files.